# Batching raw ultrasound data with `zea.Dataloader`

`zea.Dataloader` turns a folder of zea files into a batched, shuffled, multi-threaded stream of
samples, ready for a training loop. It is built on [Google Grain](https://github.com/google/grain).

We'll be using channel data (`raw_data`) in this notebook. We'll stream from the
[TU/e carotid dataset](https://huggingface.co/datasets/zeahub/zea-carotid-2023) on the Hugging Face
Hub. `zea.File` streams and caches only the HDF5 chunks a sample actually touches.

See the [`Dataloader` API reference](../../_autosummary/zea.data.dataloader.rst) for the full set of
arguments.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tue-bmd/zea/blob/main/docs/source/notebooks/data/zea_dataloader_example.ipynb)
&nbsp;
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/tue-bmd/zea/blob/main/docs/source/notebooks/data/zea_dataloader_example.ipynb)
&nbsp;
[![Hugging Face dataset](https://img.shields.io/badge/Hugging%20Face-Dataset-yellow?logo=huggingface)](https://huggingface.co/datasets/zeahub/zea-carotid-2023)

‼️ **Important:** This notebook is optimized for **GPU/TPU**. Code execution on a **CPU** may be very slow.

If you are running in Colab, please enable a hardware accelerator via:

**Runtime → Change runtime type → Hardware accelerator → GPU/TPU** 🚀.

In [1]:
%%capture
%pip install zea

In [2]:
file_paths = [
    "hf://zeahub/zea-carotid-2023/data/2_long_2cm_0000.hdf5",
    "hf://zeahub/zea-carotid-2023/data/3_long_2cm_L_0000.hdf5",
    "hf://zeahub/zea-carotid-2023/data/3_cross_2cm_L_0000.hdf5",
]
plane_waves = list(range(128, 149))  # the plane-wave transmits of each frame

In [3]:
import os

os.environ["KERAS_BACKEND"] = "jax"

In [4]:
import zea
from zea.visualize import set_mpl_style

zea: Using backend 'jax'


In [5]:
zea.init_device(verbose=False)
set_mpl_style()

## 1. Batching raw channel data 📀

Point `zea.Dataloader` at the file(s) or folder(s), and give it the key to read. Each carotid file holds
`raw_data` of shape `(150, 149, 2176, 128, 1)` — `(n_frames, n_tx, n_ax, n_el, n_ch)` — where every
frame is a sweep of 128 focused scanlines followed by 21 plane waves. `axis_selections` loads only
the plane waves, saving the memory and I/O of the scanlines we don't need here.

In [6]:
loader = zea.Dataloader(
    file_paths,
    key="data/raw_data",
    batch_size=2,
    limit_n_frames=1,  # keeps this notebook fast
    axis_selections={1: plane_waves},
)

print("batches:", len(loader))

batch = next(iter(loader))
print("first batch:", batch.shape, batch.dtype)
loader.close()

zea: Loading cached result for _find_h5_file_shapes.
batches: 2
first batch: (2, 21, 2176, 128, 1) int16


## 2. Temporal blocks ⏰

Set `n_frames > 1` to get blocks of consecutive frames per sample — useful for models that see
short sequences. For raw data, put the frame axis first with `frame_axis=0` so samples keep the
`(n_frames, n_tx, n_ax, n_el, n_ch)` layout of the file.

**More options**
- `frame_index_stride` subsamples within a block
- `overlapping_blocks=True` slides the block by one frame instead of `n_frames`.

In [7]:
loader = zea.Dataloader(
    file_paths,
    key="data/raw_data",
    batch_size=None,  # no batching: one sample at a time
    n_frames=2,
    frame_axis=0,
    axis_selections={1: plane_waves},
    shuffle=False,
)

# We're not iterating this dataloader to avoid (pre)fetching all frames.
print("blocks:", len(loader))
print("sample shape:", loader.shape)
loader.close()

zea: Loading cached result for _find_h5_file_shapes.
blocks: 225
sample shape: (2, 21, 2176, 128, 1)


## 3. Metadata and file selection 📊

A zea file carries **much more** than its data array! 

`return_metadata` asks the loader to return a `(sample, metadata)` tuple instead of a bare array, and `file_filter` drops files before any frames are indexed. Both take **dotted paths** into the file: a path may point at a single field (`"scan.sound_speed"`) or at a whole group (`"metadata.subject"`), in which case everything below it is loaded.

`file_filter` maps such paths to conditions — a plain value for equality, a callable for anything
else, or the `EXISTS` helper to require that a field is present at all — and ANDs them. Below we
keep only subject `3`, which leaves two of the three files.

The returned metadata mirrors the [zea file spec](../../_autosummary/zea.data.spec.rst).

> **Note:** when batching is on, metadata is stacked leaf by leaf just like the data, so every file
> must supply the same fields with the same shapes. If your metadata varies in shape between files,
> use `batch_size=None` and batch it yourself.


In [8]:
loader = zea.Dataloader(
    file_paths,
    key="data/raw_data",
    batch_size=None,
    limit_n_frames=2,
    axis_selections={1: plane_waves},
    shuffle=False,
    file_filter={"metadata.subject.id": "3"},
    return_metadata=[
        "metadata.subject",
        "metadata.annotations",
        "scan.sound_speed",
        "scan.time_to_next_transmit",
        "probe.probe_geometry",
        "us_machine",
    ],
)

print("samples after filtering:", len(loader))
sample, metadata = next(iter(loader))


def print_tree(tree, indent=""):
    for key, value in tree.items():
        if isinstance(value, dict):
            print(f"{indent}{key}/")
            print_tree(value, indent + "  ")
        else:
            shape = getattr(value, "shape", ())
            print(f"{indent}{key}: {value if shape == () else f'{value.dtype} {shape}'}")


print_tree(metadata)

zea: DEBUG file_filter excluded 'hf://zeahub/zea-carotid-2023/data/2_long_2cm_0000.hdf5'.


zea: file_filter kept 2/3 files (1 removed).
zea: Loading cached result for _find_h5_file_shapes.
samples after filtering: 4
metadata/
  subject/
    id: 3
    type: human
  annotations/
    anatomy: carotid artery
    view: Longitudinal section 2cm from bifurcation (left)
scan/
  sound_speed: 1540.0
  time_to_next_transmit: float32 (149,)
probe/
  probe_geometry: float32 (128, 3)
us_machine: Verasonics Vantage 256
file/
  fullpath: hf://zeahub/zea-carotid-2023/data/3_long_2cm_L_0000.hdf5
  filename: 3_long_2cm_L_0000
  indices: (0, [128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148], slice(None, 2176, None), slice(None, 128, None), slice(None, 1, None))
